In [0]:
from pyspark.sql.functions import col,lit,current_timestamp,max,date_trunc
from delta.tables import DeltaTable
from datetime import datetime,UTC
import uuid

In [0]:
%sql

USE CATALOG novadb_lakehouse;
CREATE SCHEMA IF NOT EXISTS bronze;

<h3> Bronze Ingestion Tracker Delta Table </h3>

In [0]:
%sql

CREATE TABLE IF NOT EXISTS novadb_lakehouse.bronze.Ingestion_control(
    Layer STRING,
    Tblname STRING,
    ts_col STRING,
    pk_col STRING,
    last_successful_ts TIMESTAMP,
    last_successful_pk BIGINT,
    last_runid STRING,
    rows_written BIGINT,
    run_status STRING,
    Updated_at TIMESTAMP
) USING DELTA

In [0]:

def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("novadb_lakehouse.bronze.ingestion_control") \
        .filter(
            (col("Layer") == "bronze") &
            (col("Tblname") == table_name) &
            (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

 

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):

    control_df = spark.createDataFrame(
        [
            {
                "Layer": "bronze",
                "Tblname": table_name,
                "ts_col": ts_col,
                "pk_col": pk_col,
                "last_successful_ts": last_ts,
                "last_successful_pk": int(last_pk) if last_pk is not None else None,
                "last_runid": run_id,
                "rows_written": int(rows_written),
                "run_status": "success",
                "Updated_at": datetime.now(UTC)
            }
        ],

        schema= """
        Layer STRING,
        Tblname STRING,
        ts_col STRING,
        pk_col STRING,
        last_successful_ts TIMESTAMP,
        last_successful_pk BIGINT,
        last_runid STRING,
        rows_written BIGINT,
        run_status STRING,
        Updated_at TIMESTAMP
        """
    )

    target_control_df = DeltaTable.forName(spark, "novadb_lakehouse.bronze.ingestion_control")

    target_control_df.alias("target").merge(
        control_df.alias("source"),
        "target.Tblname = source.Tblname AND target.Layer = source.Layer"
    ).whenMatchedUpdate(set = {
        "ts_col": "source.ts_col",
        "pk_col": "source.pk_col",
        "last_successful_ts": "source.last_successful_ts",
        "last_successful_pk": "source.last_successful_pk",
        "last_runid": "source.last_runid",
        "rows_written": "source.rows_written",
        "run_status": "source.run_status",
        "updated_at": "source.updated_at"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()





<h3>Flow Diagram </h3>


![image_1782019410726.png](./image_1782019410726.png "image_1782019410726.png")

In [0]:
table_config = {
"orders": {"pk_col": "order_id", "ts_col": "updated_at"},
"payments": {"pk_col": "payment_id", "ts_col": "processed_at"},
"products": {"pk_col": "product_id", "ts_col": "updated_at"},
}

bronze_run_id = str (uuid.uuid4())
print("Current Bronze Runid", bronze_run_id)

In [0]:
for table_name,cfg in table_config.items():

    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"novacartdb_azuresql_conn_catalog.dbo.{table_name}"
    target_table = f"novadb_lakehouse.bronze.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    if(last_successful_ts is not None):
      last_successful_ts = last_successful_ts.replace(microsecond=(last_successful_ts.microsecond // 1000)* 1000)

    print(f"============= Processing {table_name} =========")
    print("last_successful_ts",last_successful_ts)
    print("last_successful_pk",last_successful_pk)

    source_df = spark.read.table(source_table) \
                .withColumn(ts_col, date_trunc("MILLISECOND",col(ts_col).cast("timestamp")))

    if last_successful_ts is None:
        rows_to_load = source_df
    elif last_successful_pk is None:
        rows_to_load = source_df.filter(col(ts_col) >= lit(last_successful_ts))
    else:
        """
        This condition handles rows having the same timestamp as the last processed record.
        Since multiple records can share the same timestamp, the primary key acts as a
        tie-breaker to load only the remaining unprocessed rows and avoid duplicates/missed records.
        """
        rows_to_load = source_df.filter(
            (col(ts_col) >= lit(last_successful_ts)) &
            (col(pk_col).cast("long") > lit(int(last_successful_pk)))
            )
    

    rows_to_load = ( rows_to_load \
                    .withColumn("bronze_ingested_at",current_timestamp()) \
                    .withColumn("bronze_run_id", lit(bronze_run_id)) \
                    .withColumn("bronze_source_table",lit(source_table))
                    )
    

    rows_count = rows_to_load.count()

    print("Rows to load ", rows_count)

    if rows_count == 0:
        upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,rows_count,bronze_run_id)

        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table) #append /

    print(f"Wrote data to {target_table}")
    
    watermark_row = (
        rows_to_load
        .select(
            col(ts_col).cast("timestamp").alias("ts"),
            col(pk_col).cast("long").alias("pk")
        )
        .orderBy(col("ts").desc(), col("pk").desc())
        .limit(1)
        .collect()
    )

    max_ts = watermark_row[0]["ts"]
    max_pk = watermark_row[0]["pk"]
    upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,rows_count,bronze_run_id)
    



In [0]:
%sql

SELECT * FROM novadb_lakehouse.bronze.ingestion_control;
